In [ ]:
%run ./expectations

In [ ]:
NOTEBOOK = "tests/gold"
errors = []

fact_aire = spark.table(f"{GOLD_TABLE}.fact_calidad_aire")
fact_trafico = spark.table(f"{GOLD_TABLE}.fact_trafico")
fact_diario = spark.table(f"{GOLD_TABLE}.fact_trafico_diario")
dim_estacion = spark.table(f"{GOLD_TABLE}.dim_estacion_aire")
dim_punto = spark.table(f"{GOLD_TABLE}.dim_punto_trafico")
dim_distrito = spark.table(f"{GOLD_TABLE}.dim_distrito")
dim_fecha = spark.table(f"{GOLD_TABLE}.dim_fecha")
dim_magnitud = spark.table(f"{GOLD_TABLE}.dim_magnitud")

In [ ]:
# Grain declared in gold/_catalog.yml
errors += [c for c in [
    expect_unique("aire_grain", fact_aire, ["estacion", "fecha", "magnitud"], NOTEBOOK),
    expect_unique("trafico_grain", fact_trafico, ["id", "fecha"], NOTEBOOK),
    expect_unique("trafico_diario_grain", fact_diario, ["id", "fecha"], NOTEBOOK),
    expect_unique("estacion_pk", dim_estacion, ["codigo_corto"], NOTEBOOK),
    expect_unique("punto_pk", dim_punto, ["id"], NOTEBOOK),
    expect_unique("distrito_pk", dim_distrito, ["cod_dis"], NOTEBOOK),
    expect_unique("fecha_pk", dim_fecha, ["fecha"], NOTEBOOK),
    expect_unique("magnitud_pk", dim_magnitud, ["codigo"], NOTEBOOK),
] if c]

In [ ]:
# Value ranges. Invalid readings (validez='N') are kept on purpose and may be negative
errors += [c for c in [
    expect("aire_dato_no_negativo", fact_aire,
           "validez <> 'V' OR dato IS NULL OR dato >= 0", NOTEBOOK),
    expect("aire_fecha_no_futura", fact_aire, "fecha <= current_date()", NOTEBOOK),
    expect("aire_validez", fact_aire, "validez IN ('V', 'N')", NOTEBOOK),
    expect("trafico_intensidad", fact_trafico,
           f"intensidad IS NULL OR intensidad BETWEEN 0 AND {INTENSIDAD_MAX}", NOTEBOOK),
    expect("diario_lecturas_ok", fact_diario, "lecturas_ok <= lecturas", NOTEBOOK),
    expect("diario_lecturas", fact_diario, "lecturas > 0", NOTEBOOK),
] if c]

In [ ]:
# lpad(2) is what every district join relies on
errors += [c for c in [
    expect("distrito_cod_len", dim_distrito, "length(cod_dis) = 2", NOTEBOOK),
    expect("punto_distrito_len", dim_punto, "distrito IS NULL OR length(distrito) = 2", NOTEBOOK),
    expect("estacion_coords", dim_estacion,
           "latitud IS NOT NULL AND longitud IS NOT NULL", NOTEBOOK),
] if c]

In [ ]:
# Rows the inner joins in gold/facts would drop without raising
errors += [c for c in [
    expect_no_orphans("aire_estacion_fk", spark.table(f"{SILVER_TABLE}.aire"), "estacion",
                      dim_estacion, "codigo_corto", NOTEBOOK),
    expect_no_orphans("trafico_punto_fk", spark.table(f"{SILVER_TABLE}.trafico"), "id",
                      dim_punto, "id", NOTEBOOK),
] if c]

In [ ]:
report(errors, NOTEBOOK)